# Dataloading guide

This notebook shows how to make the dataloaders, and how to call up one specific patientID

In [7]:
# imports 
import sys
import os
from pathlib import Path
path_Project = Path(os.getcwd()).parent.parent
path_Git = path_Project.parent

%load_ext autoreload
%autoreload 2

import torch

from src.config_presets.tools.get_config import get_config
from src.dataset.load_dataset import load_dataset, load_dataset_total
from src.models.tools.save_model import save_model
from src.training.train_multi import train
from src.utils.logging.logging import setup_logging
from src.utils.parse_args import parse_args
from src.utils.set_random_seed import set_random_seed
from src.hyper_opt.hyperHandler import HyperTuning_Handler
from src.utils.fileHandler import create_file, create_textfile


from src.utils.loss_func.get_loss_function import get_loss_function
from src.dataset.get_dataloader import make_dataloader   
from src.dataset.get_transforms import get_transforms


# config stuff
config = get_config('Multi_tox')

# Disable randomness
set_random_seed(config['general']['seed'])

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'src'

In [3]:
# you can load a K-fold dataset like this:
DFs_col = load_dataset_total(config)
trainDF_col = DFs_col[0]
valDF_col= DFs_col[1]
testDF_col = DFs_col[2]


# make the training and validation transforms
train_transforms, val_transforms = get_transforms(config)

# make the dataloaders
train_loader, metadata = make_dataloader(config, trainDF_col[0], train_transforms, validation_mode=False)
val_loader, _ = make_dataloader(config, valDF_col[0], val_transforms, validation_mode=True)

NameError: name 'load_dataset_total' is not defined

# Call up a specific patient

Now, each 'dataset' is given a list of patientIDs for all of the patients it contains. This is a list of strings under the variable name `patient_IDs_list`, which, for example, can be accessed by `val_dataset.patient_IDs_list`.

The 'dataloader' now has a built-in function which can use this list to return the data for a specific patient (by using the indexing of that list, NOTE: make sure shuffling is disabled on the dataloader). It can be used as follows:

In [8]:
batch_data = val_loader.get_patient('9428962')   # enter the patient ID string to get the data of the patient, this function will do all of the work for you

NameError: name 'val_loader' is not defined

In [ ]:
# to put the data through the model proabably looks smth like this:

def move_batch_to_device(batch, device):
    # batch is a dictionary
    inputs, clinical_features, targets = batch['input'], batch['features'], batch['label_list']
    inputs = inputs.to(device=device)
    clinical_features = clinical_features.to(device=device)
    targets = targets.to(device=device)
    return inputs, clinical_features, targets

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

inputs, clinical_features, targets = move_batch_to_device(batch_data, DEVICE)

# Make predictions
outputs = model(x=inputs, features=clinical_features)